In [1]:
import numpy as np
import matplotlib.pyplot as plt
import importlib

In [10]:
data_original = np.load(r"F:\temp_compare\data_original.npz", allow_pickle=True)
data_modular = np.load(r"F:\temp_compare\data_modular.npz", allow_pickle=True)

In [11]:
X_train_original = data_original['X_trainT']
X_test_original = data_original['X_testT']
y_train_original = data_original['y_trainT']
y_test_original = data_original['y_testT']
X_train_modular = data_modular['X_train']
X_test_modular = data_modular['X_test']
y_train_modular = data_modular['y_train']
y_test_modular = data_modular['y_test']

In [1]:
import torch
import utils
from pathlib import Path
import preprocess_data as ppd
from process_session import session as ss
import process_probe as pp
import process_attribution as pa

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

session_id = 1047969464
session_obj = ss(session_id)

spikes_obj = ppd.pre_process_spikes(session_obj.units, session_obj.spike_times, bin_size=0.004, sigma=3)
spikes_obj.getSpkMat(session_obj.active_times[0],session_obj.active_times[1])
spikes_obj.convolve_with_gaussian()
spikes_obj.zscore()

probe_obj = pp.probe(session_obj)
lfp_obj = ppd.pre_process_lfp(probe_obj.lfp, 1250)
lfp_obj.filter_lfp(probe_obj.bands)

input_size = spikes_obj.spkMat.shape[1]
hidden_size = 50
num_layers = 1
seqlength = 750
num_epochs = 15
X_train, y_train, X_test, y_test = ppd.generate_training_data(lfp_obj, spikes_obj, seqlength)

# np.savez_compressed('F:/temp_compare/data_modular.npz', X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test)

# models, lossesAll = utils.train_models(probe_obj, input_size, hidden_size, num_layers, seqlength, device, num_epochs, X_train, y_train)
# output_dir = Path('E:/vbn_s3_cache')
# utils.save_models(models, output_dir, session_id)

# check if a variable models exists otherwise load it from the saved models
# output_dir = Path(r"Z:\Buzsakilabspace\LabShare\NoamNitzan\Open_Access\Allen_2022")
# args = [input_size, hidden_size, num_layers, seqlength, device]
# models = utils.load_models(output_dir, session_id, probe_obj, args)

# output_dir = Path(r'F:\vbn_s3_cache')
# dur = 720
# bin_size = 0.004
# pa.divide_task_for_attr(models, session_id, output_dir, spikes_obj, bin_size, probe_obj, dur)

c:\Users\Marisol\anaconda3\envs\allensdk\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 922/922 [02:05<00:00,  7.36it/s]
c:\Users\Marisol\anaconda3\envs\allensdk\lib\site-packages\hdmf\utils.py:668: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.5.1 because version 1.8.0 is already loaded.
  return func(args[0], **pargs)
c:\Users\Marisol\anaconda3\envs\allensdk\lib\site-packages\hdmf\utils.py:668: UserWarning: Ignoring cached namespace 'core' version 2.5.0 because version 2.6.0-alpha is already loaded.
  return func(args[0], **pargs)
c:\Users\Marisol\anaconda3\envs\allensdk\lib\site-packages\hdmf\utils.py:668: UserWarning: Ignoring cached namespace 'hdmf-experimental' version 0.2.0 because version 0.5.0 is already loaded.
  return func(args[0], **pargs)
100%|██████████| 8/8 [07:06

In [2]:
import numpy as np
data_original = np.load(r"F:\temp_compare\data_og.npz", allow_pickle=True)
X_train_original = data_original['X_trainT']
X_test_original = data_original['X_testT']
y_train_original = data_original['y_trainT']
y_test_original = data_original['y_testT']

In [9]:
X_train_original

array([[-0.6076582 , -0.20017773, -0.21072865, ..., -0.18429613,
         1.0256703 , -0.08600001],
       [-0.6076582 , -0.20017773, -0.21072865, ..., -0.18429613,
         1.6730132 , -0.2908631 ],
       [-0.6076582 , -0.20017773, -0.21072865, ..., -0.18429613,
         2.3163424 , -0.37238628],
       ...,
       [-0.60547405, -0.20017773,  1.0879548 , ..., -0.18429613,
        -0.33618033, -0.48987877],
       [-0.60064423, -0.20017773,  0.49412853, ..., -0.18429613,
        -0.33618033, -0.2455351 ],
       [-0.58750314, -0.20017773,  0.13160059, ..., -0.18429613,
        -0.33618033,  0.00526426]], dtype=float32)

In [3]:
np.allclose(X_train_original, X_train), np.allclose(y_train_original, y_train), np.allclose(X_test_original, X_test), np.allclose(y_test_original, y_test)

(True, True, True, True)

In [ ]:
import torch
X_train_original = torch.Tensor(data_original['X_trainT']).float()
X_test_original = torch.Tensor(data_original['X_testT']).float()
y_train_original = torch.Tensor(data_original['y_trainT']).float()
y_test_original = torch.Tensor(data_original['y_testT']).float()
(X_train_original == X_train).all(), (X_test_original == X_test).all(), (y_train_original == y_train).all(), (y_test_original == y_test).all()

(tensor(True), tensor(True), tensor(True), tensor(True))

In [4]:
import blosc2
import numpy as np

In [78]:
a_noam = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band1_noam.npy")
a_mine = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band1.npy")
print(np.mean(a_noam), np.mean(a_mine))
print(np.sum(a_noam), np.sum(a_mine))
print(np.std(a_noam), np.std(a_mine))
print(np.allclose(np.mean(a_noam, axis=0), np.mean(a_mine, axis=0)))

0.00055240013 0.0005524001
73679.13 73679.125
0.09417887 0.0941789
True


In [79]:
a_noam = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band2_noam.npy")
a_mine = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band2.npy")
print(np.mean(a_noam), np.mean(a_mine))
print(np.sum(a_noam), np.sum(a_mine))
print(np.std(a_noam), np.std(a_mine))
print(np.allclose(np.mean(a_noam, axis=0), np.mean(a_mine, axis=0)))

0.000684496 0.000684496
91298.08 91298.08
0.020936504 0.020936504
True


In [80]:
a_noam = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band3_noam.npy")
a_mine = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band3.npy")
print(np.mean(a_noam), np.mean(a_mine))
print(np.sum(a_noam), np.sum(a_mine))
print(np.std(a_noam), np.std(a_mine))
print(np.allclose(np.mean(a_noam, axis=0), np.mean(a_mine, axis=0)))

0.002583404 0.002583404
344574.44 344574.44
1.8097694 1.8097694
True


In [81]:
a_noam = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band4_noam.npy")
a_mine = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band4.npy")
print(np.mean(a_noam), np.mean(a_mine))
print(np.sum(a_noam), np.sum(a_mine))
print(np.std(a_noam), np.std(a_mine))
print(np.allclose(np.mean(a_noam, axis=0), np.mean(a_mine, axis=0)))

0.0010266793 0.0010266793
136938.48 136938.48
0.18416052 0.18416052
True


In [82]:
a_noam = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band5_noam.npy")
a_mine = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band5.npy")
print(np.mean(a_noam), np.mean(a_mine))
print(np.sum(a_noam), np.sum(a_mine))
print(np.std(a_noam), np.std(a_mine))
print(np.allclose(np.mean(a_noam, axis=0), np.mean(a_mine, axis=0)))

0.0006470854 0.0006470854
86308.26 86308.26
0.07330191 0.073301904
True
